# Prompt Templates & Management 📝🗂️

Welcome to Prompt Templates. As you transition from writing quick personal scripts to shipping a real software product powered by GenAI, hardcoding long instruction strings directly inside your application source code becomes a massive engineering anti-pattern.

If you need to tweak an instruction or fix a typo, you shouldn't have to re-deploy your entire backend microservice. Prompt Templates solve this by separating your prompt logic from your application code—treating prompts like configuration files or parameterized functions.

## Phase 1: The Developer Analogy & Why We Need Templates
### The Anti-Pattern (Hardcoded Strings):

In [ ]:
# Don't do this in a production codebase!
def generate_code_review(language, code):
    prompt = "You are a " + language + " expert. Review this code for bugs: " + code
    return client.chat.completions.create(model="gpt-4.1-mini", messages=[{"role": "user", "content": prompt}])

### Why this breaks at scale:

**Maintainability:** Prompts get messy, long, and filled with multiline markdown blocks. Scattering them inside business logic files makes them impossible to read.
Collaboration: Product managers, copywriters, or security teams can't review or update prompts if they are buried deep inside Python or TypeScript files.
Versioning: You can't easily track prompt revisions, run regression tests, or perform A/B testing on prompt wording.

**The Template Pattern (The Solution):**
Just like web developers use HTML templates (Jinja2, Handlebars, React JSX) to separate markup from data controllers, prompt templates use placeholder variables ({variable}) that are populated at runtime.  

## Phase 2: Standardizing with LangChain / Core Frameworks
In modern AI development, standard libraries like langchain-core provide robust abstractions for prompt templates.
There are two primary types of templates you will use:
**String Prompt Templates (PromptTemplate):** Best for single-turn instructions, text completion, or base string formatting.  
**Chat Prompt Templates (ChatPromptTemplate):** Best for modern multi-role chat APIs (system, user, assistant), allowing you to template individual message blocks dynamically.  

## Phase 3: Python Code Implementation
Here is how you implement production-grade prompt templating using langchain-core, separating your text layout from your execution code.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from openai import OpenAI

client = OpenAI()

# 1. Define a structured Chat Prompt Template with roles and placeholder variables
code_review_template = ChatPromptTemplate.from_messages([
    ("system", "You are an elite {language} security auditor. Find vulnerabilities and output markdown lists."),
    ("user", "Please review the following module for security bugs:\n\n```_lang\n{source_code}\n```")
])

# 2. Format the template dynamically at runtime by passing keyword arguments
formatted_messages = code_review_template.format_messages(
    language="Python",
    source_code="user_input = request.GET.get('q')\neval(user_input)"
)

# 3. Pass the structured template messages straight into your LLM SDK client
response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[{"role": m.type, "content": m.content} for m in formatted_messages],
    temperature=0.0
)

print(response.choices[0].message.content)

## Phase 4: Production Prompt Management Best Practices
As your application expands, scale your prompt architecture using these engineering standards:

**Externalize Prompt Storage:** Store your prompts outside of your code repository's main logic files—either as standalone .yaml / .json files, or inside a dedicated prompt registry/management tool (like LangSmith, Portkey, or Langfuse).

**Version Control:** Treat prompt files like database migrations or API contracts. If you change a prompt template, track it via Git commits so you can trace regressions if user outputs suddenly drift.

**Strict Validation:** Ensure your backend code validates input variables before passing them into a template to prevent missing variable key errors or template injection bugs.